# Análisis MongoDB — Sharding + Búsqueda Textual
**BD2 Lab 16: MongoDB Distribuido vs Citus**

Dataset: News Headlines About President Milei (2,224 registros únicos)

**Nota:** Este notebook funciona sin conexión a MongoDB para tablas/gráficos. 
Para las queries con `explain("executionStats")` requiere el cluster MongoDB activo.

In [ ]:
import pandas as pd
import json
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np

DATA_FILE = r"../../dataset/processed/milei_news_clean.json"
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "news_analysis"
COLLECTION = "milei_news"

df = pd.read_json(DATA_FILE)
print(f"Registros cargados: {len(df)}")
df.head(3)

---
## P2: Estadísticas del Dataset Limpio

In [ ]:
print("=== TABLA DE ESTADÍSTICAS ===")
print(f"Registros totales: {len(df)}")
print(f"Secciones únicas: {df['section'].nunique()}")
print(f"Periódicos únicos: {df['news_paper'].nunique()}")
print(f"Registros sin summary: {(df['summary'] == '').sum()}")
print(f"Registros sin fecha: {df['published'].isna().sum()}")
print(f"\nDistribución de secciones (top 15):")
print(df['section'].value_counts().head(15).to_string())
print(f"\nDistribución de periódicos:")
print(df['news_paper'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_sections = df['section'].value_counts().head(15)
axes[0].barh(range(len(top_sections)), top_sections.values)
axes[0].set_yticks(range(len(top_sections)))
axes[0].set_yticklabels(top_sections.index)
axes[0].set_xlabel('Noticias')
axes[0].set_title('Top 15 Secciones')

top_papers = df['news_paper'].value_counts().head(10)
axes[1].barh(range(len(top_papers)), top_papers.values)
axes[1].set_yticks(range(len(top_papers)))
axes[1].set_yticklabels(top_papers.index)
axes[1].set_xlabel('Noticias')
axes[1].set_title('Top 10 Periódicos')

plt.tight_layout()
plt.show()

---
## P3: Análisis de Cardinalidad de section (para elegir shard key)

section tiene ~70 valores distintos con distribución sesgada (Política domina).
Conclusión: **shard key hash** es la estrategia correcta para balancear chunks.

In [ ]:
section_dist = df['section'].value_counts()
print(f"Cardinalidad de section: {len(section_dist)}")
print(f"Sección más frecuente: '{section_dist.index[0]}' con {section_dist.iloc[0]} registros ({section_dist.iloc[0]/len(df)*100:.1f}%)")
print(f"Sección menos frecuente: '{section_dist.index[-1]}' con {section_dist.iloc[-1]} registros")
print(f"\nRatio max/min: {section_dist.iloc[0] / max(section_dist.iloc[-1], 1):.0f}x")
print(f"\nConclusión: Distribución sesgada -> se elige HASH sharding para balancear chunks")

---
## Q1–Q5: Implementación y medición de tiempos

### Requiere conexión a MongoDB (cluster activo)
Si no hay conexión, saltará al bloque offline.

In [ ]:
MONGO_AVAILABLE = False
try:
    from pymongo import MongoClient
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    db = client[DB_NAME]
    collection = db[COLLECTION_NAME]
    MONGO_AVAILABLE = True
    print("Conexión a MongoDB exitosa.")
except Exception as e:
    print(f"No se pudo conectar a MongoDB: {e}")
    print("Las queries requerirán el cluster activo para ejecutarse.")

### Q1: Búsqueda textual múltiples términos

In [ ]:
if MONGO_AVAILABLE:
    import time
    query = {"$text": {"$search": "dolar inflacion"}}
    
    start = time.time()
    results = list(collection.find(query, {"score": {"$meta": "textScore"}}).sort([("score", {"$meta": "textScore"})]).limit(20))
    elapsed = time.time() - start
    
    # explain
    explain = collection.find(query).explain()
    
    print(f"Q1 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"Q1 - Resultados: {len(results)}")
    print(f"\nTop 5 resultados:")
    for r in results[:5]:
        score = r.get('score', 0)
        print(f"  [{score:.3f}] {r.get('title', '')[:70]} - {r.get('section', '')}")
    print(f"\n--- EXPLAIN (executionStats) ---")
    import json as _json
    print(_json.dumps(explain.get('executionStats', {}), indent=2, default=str)[:2000])
else:
    print("Q1: Requiere conexión MongoDB para ejecutar.")

### Q2: Búsqueda con exclusión

In [ ]:
if MONGO_AVAILABLE:
    import time
    query = {"$text": {"$search": "Milei -Twitter"}}
    
    start = time.time()
    results = list(collection.find(query).sort("published", -1).limit(20))
    elapsed = time.time() - start
    
    explain = collection.find(query).sort("published", -1).explain()
    
    print(f"Q2 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"Q2 - Resultados: {len(results)}")
    print(f"\nTop 5 resultados:")
    for r in results[:5]:
        print(f"  {r.get('published', '')} - {r.get('title', '')[:60]}")
    print(f"\n--- EXPLAIN (executionStats) ---")
    import json as _json
    print(_json.dumps(explain.get('executionStats', {}), indent=2, default=str)[:2000])
else:
    print("Q2: Requiere conexión MongoDB.")

### Q3: Top 10 por fecha

In [ ]:
if MONGO_AVAILABLE:
    import time
    start = time.time()
    results = list(collection.find().sort("published", -1).limit(10))
    elapsed = time.time() - start
    
    explain = collection.find().sort("published", -1).limit(10).explain()
    
    print(f"Q3 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"\nTop 10 más recientes:")
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r.get('published', '')} - {r.get('title', '')[:60]}")
    print(f"\n--- EXPLAIN (executionStats) ---")
    import json as _json
    print(_json.dumps(explain.get('executionStats', {}), indent=2, default=str)[:2000])
else:
    print("Q3: Requiere conexión MongoDB.")

### Q4: Agregación por shard key (section)

In [ ]:
if MONGO_AVAILABLE:
    import time
    pipeline = [
        {"$group": {"_id": "$section", "count": {"$sum": 1}}},
        {"$sort": {"count": -1}}
    ]
    
    start = time.time()
    results = list(collection.aggregate(pipeline))
    elapsed = time.time() - start
    
    explain = db.command("aggregate", COLLECTION, pipeline=pipeline, explain=True)
    
    print(f"Q4 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"\nSecciones (top 10):")
    for r in results[:10]:
        print(f"  {r['_id']}: {r['count']}")
    print(f"\n--- EXPLAIN ---")
    import json as _json
    print(_json.dumps(explain, indent=2, default=str)[:2000])
else:
    print("Q4: Requiere conexión MongoDB.")

### Q5: Agregación por atributo no particionado (news_paper)

In [ ]:
if MONGO_AVAILABLE:
    import time
    pipeline = [
        {"$group": {"_id": "$news_paper", "count": {"$sum": 1}}},
        {"$sort": {"count": -1}}
    ]
    
    start = time.time()
    results = list(collection.aggregate(pipeline))
    elapsed = time.time() - start
    
    explain = db.command("aggregate", COLLECTION, pipeline=pipeline, explain=True)
    
    print(f"Q5 - Tiempo: {elapsed*1000:.2f} ms")
    print(f"\nPeriódicos:")
    for r in results:
        print(f"  {r['_id']}: {r['count']}")
    print(f"\n--- EXPLAIN ---")
    import json as _json
    print(_json.dumps(explain, indent=2, default=str)[:2000])
else:
    print("Q5: Requiere conexión MongoDB.")

### Q1–Q5: Tabla resumen de tiempos

In [ ]:
if MONGO_AVAILABLE:
    results_table = pd.DataFrame({
        "Query": ["Q1 (text search)", "Q2 (exclusion)", "Q3 (top10 fecha)", "Q4 (group by section)", "Q5 (group by news_paper)"],
        "Tiempo (ms)": [q1_t, q2_t, q3_t, q4_t, q5_t],  # reemplazar con variables reales
        "Resultados": [20, 20, 10, 70, 20]
    })
    print("Tabla de tiempos Q1-Q5")
    print("Completar con las variables de tiempo de las celdas anteriores.")
    # Una vez ejecutadas las celdas Q1-Q5, descomentar:
    # print(tiempos_df.to_string())
else:
    print("Tabla de tiempos: pendiente (ejecutar con MongoDB activo)")

---
## P4: Análisis del índice de texto en MongoDB

El índice de texto en MongoDB:
- Usa una estructura de **índice invertido** (inverted index)
- Implementa **stemming** basado en el idioma configurado (`default_language: "spanish"`)
- No usa TF-IDF directamente; el `textScore` es un cálculo interno de relevancia que considera:
  - Frecuencia del término en el documento
  - Rareza del término en la colección
- La búsqueda textual en sharding se distribuye: cada shard ejecuta la búsqueda localmente y mongos mergea los resultados ordenados por `textScore`

Ver extracto de `explain("executionStats")` en las celdas Q1-Q2 para ver:
- `executionStages.stage` debe ser `SHARD_MERGE`
- `allShards` con estadísticas por shard

In [ ]:
# Resumen P4
print("Extraer de explain anterior:")
print("- executionStages.stage (SHARD_MERGE)")
print("- shards[] con executionStats por shard")
print("- totalDocsExamined vs totalDocsReturned")

---
## P1: Comandos para verificar sharding

Ejecutar en MongoDB Compass o mongosh:
```javascript
sh.status()
db.milei_news.getShardDistribution()
db.milei_news.stats()
```

Capturar screenshots y pegarlos en `mongodb/screenshots/`

In [ ]:
print("=== Resumen para informe ===")
print(f"Registros totales: {len(df)}")
print(f"Secciones: {df['section'].nunique()}")
print(f"Periódicos: {df['news_paper'].nunique()}")
print(f"Shard key estrategia: HASH sobre section")